[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nasaharvest/crop-stage-detection/blob/main/examples/02_gee.ipynb)


In [ ]:
!git clone https://github.com/nasaharvest/crop-stage-detection.git /content/crop-stage-detection
%cd /content/crop-stage-detection/examples
!pip install -q -r ../requirements.txt -r ../requirements-gee.txt


# Example 2 — Full Pipeline with Google Earth Engine

`run_crop_stage_from_gee` is the simplest entry point for GEE users: provide a
single polygon and it returns the crop stage for the last available observation.

This notebook starts with that simple one-polygon workflow. Later sections show
how to visualize the NDVI time series for that polygon and process multiple
fields in a simple batch example.

### Prerequisites

1. A GEE account — sign up at https://earthengine.google.com
2. Install both requirements files: `pip install -r requirements.txt -r requirements-gee.txt`
3. Authenticate: the next cell calls `ee.Authenticate()`, which opens an OAuth
   flow the first time (or whenever credentials expire) — follow the prompt
   and paste back the verification code.

Call `ee.Authenticate()` and `ee.Initialize()` **before** importing `gee_fetch`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import ee
ee.Authenticate()  # opens an OAuth flow; only prompts if not already authenticated
ee.Initialize()    # <-- must come before gee_fetch import

from gee_fetch import run_crop_stage_from_gee

## 1. Run the workflow for one polygon

Start with a simple polygon and call `run_crop_stage_from_gee`.

In [ ]:
polygon = [
    [-77.897937, 35.571295],
    [-77.898152, 35.570355],
    [-77.897400, 35.569852],
    [-77.895398, 35.569557],
    [-77.895062, 35.570803],
    [-77.897937, 35.571295],  # close the ring (optional)
]

# You can optionally pass an end date; the default is today.
result = run_crop_stage_from_gee(polygon)

for k, v in result.items():
    print(f"{k:20s}: {v}")

## 2. Visualize the NDVI time series

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date, timedelta

from crop_stage import smooth_daily_interpolate_ndvi, estimate_stage_adaptive
from gee_fetch import fetch_ndvi

# Same defaults run_crop_stage_from_gee uses: 150 days of history, ending
# tomorrow (GEE's filterDate end is exclusive, so today's imagery is included).
LOOKBACK_DAYS = 150
END_DATE = (date.today() + timedelta(days=1)).isoformat()
START_DATE = (date.today() - timedelta(days=LOOKBACK_DAYS)).isoformat()

ndvi_df = fetch_ndvi(
    polygon,
    start_date=START_DATE,
    end_date=END_DATE,
    poly_name="example_field",
    buffer_m=-10,     # inset 10 m to avoid boundary pixels
)

if ndvi_df.empty:
    raise ValueError("No valid NDVI observations returned — polygon may be too cloudy, too small, or outside GEE coverage.")

df_smooth = smooth_daily_interpolate_ndvi(ndvi_df)
result = estimate_stage_adaptive(
    df_smooth["NDVI_smooth"].to_numpy(),
    dates=df_smooth["date"],
)

fig, ax = plt.subplots(figsize=(10, 4))

for sensor, grp in ndvi_df.groupby("sensor"):
    ax.scatter(grp["date"], grp["NDVI"], alpha=0.5, s=35, label=sensor)

ax.plot(df_smooth["date"], df_smooth["NDVI_smooth"], color="steelblue", lw=2, label="Smoothed")
ax.axhline(result["Lower_threshold"], color="orange", ls="--",
           label=f"Lower ({result['Lower_threshold']:.2f})")
ax.axhline(result["Upper_threshold"], color="green",  ls="--",
           label=f"Upper ({result['Upper_threshold']:.2f})")

last_obs = ndvi_df.sort_values("date").iloc[-1]
ax.scatter(last_obs["date"], last_obs["NDVI"], s=200, facecolors="none", edgecolors="red",
           linewidths=2, zorder=4, label="Current observation")

ax.set_ylim(0, 1)
ax.set_ylabel("NDVI")
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))
ax.legend(loc="upper left", fontsize=8)
ax.set_title(f"Stage: {result['Stage']} — {result['Stage_description']}")
plt.tight_layout()
plt.show()

## 3. Batch processing

This section shows two common patterns: a simple for-loop over multiple fields,
and a parallel version using ThreadPoolExecutor.

In [ ]:
import geopandas as gpd
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely.affinity import translate
from shapely.geometry import shape
import json

with open("../sample_data/sample_field.geojson") as f:
    geom1 = shape(json.load(f)["features"][0]["geometry"])
geom2 = translate(geom1, xoff=0.05)

gdf = gpd.GeoDataFrame(
    {"field_id": ["field_a", "field_b"]},
    geometry=[geom1, geom2],
    crs="EPSG:4326",
)

# Simple loop
field_results = []
for _, row in gdf.iterrows():
    field_gdf = gpd.GeoDataFrame([row], geometry="geometry", crs=gdf.crs)
    result = run_crop_stage_from_gee(field_gdf)
    field_results.append({"field_id": row["field_id"], **result})

print("Simple loop results")
print(pd.DataFrame(field_results)[["field_id", "Stage", "Stage_description", "Peak_date", "Days_since_peak"]])

# Parallel processing with ThreadPoolExecutor

def _run_field(row):
    field_gdf = gpd.GeoDataFrame([row], geometry="geometry", crs=gdf.crs)
    result = run_crop_stage_from_gee(field_gdf)
    return {"field_id": row["field_id"], **result}

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(_run_field, row) for _, row in gdf.iterrows()]
    parallel_results = [future.result() for future in as_completed(futures)]

print("\nParallel results")
print(pd.DataFrame(parallel_results)[["field_id", "Stage", "Stage_description", "Peak_date", "Days_since_peak"]])